In [1]:
import pandas as pd
import numpy as np
import xarray as xr

In [8]:
# New data provided by user corresponds to diapycnal O2 fluxes (not divergence)
# Values are in µmol m⁻² day⁻¹

flux_data = {
    "Interval": ["(1)", "(2)", "(3)", "(4)", "(5)", "(6)", "(7)"],
    "January 2020": [-1.40,-3.03,-15.39,-3.37,6.95,0.75,9.11],
    "July 2021": [-8.15,-5.48,-16.19,-2.04,2.11,3.65,5.00],
    "October 2022": [-0.18,-1.63,-1.83,-1.21,-0.65,3.89,1.51]
}

df_flux = pd.DataFrame(flux_data)

# Apply user-defined groupings
upper_jan_flux = df_flux.loc[df_flux["Interval"].isin(["(2)", "(3)", "(4)"]), "January 2020"]
lower_jan_flux = df_flux.loc[df_flux["Interval"].isin(["(5)", "(6)", "(7)"]), "January 2020"]

upper_jul_flux = df_flux.loc[df_flux["Interval"].isin(["(2)", "(3)", "(4)"]), "July 2021"]
lower_jul_flux = df_flux.loc[df_flux["Interval"].isin(["(5)", "(6)", "(7)"]), "July 2021"]

upper_oct_flux = df_flux.loc[df_flux["Interval"].isin(["(2)", "(3)", "(4)", "(5)"]), "October 2022"]
lower_oct_flux = df_flux.loc[df_flux["Interval"].isin(["(6)", "(7)"]), "October 2022"]

# Compute means
mean_fluxes = {
    "Upper Oxycline": [
        upper_jan_flux.mean(), 
        upper_jul_flux.mean(), 
        upper_oct_flux.mean()
    ],
    "Lower Oxycline": [
        lower_jan_flux.mean(), 
        lower_jul_flux.mean(), 
        lower_oct_flux.mean()
    ]
}

mean_flux_df = pd.DataFrame(mean_fluxes, index=["January 2020", "July 2021", "October 2022"])
df_flux_values = df_flux.set_index("Interval")

df_flux_values




,January 2020,July 2021,October 2022
Interval,,,
(1),-1.40,-8.15,-0.18
(2),-3.03,-5.48,-1.63
(3),-15.39,-16.19,-1.83
(4),-3.37,-2.04,-1.21
(5),6.95,2.11,-0.65
(6),0.75,3.65,3.89
(7),9.11,5.00,1.51


In [ ]:
index = ["(1)", "(2)", "(3)", "(4)", "(5)", "(6)", "(7)"]
Pt = pd.DataFrame({
    "January 2020": [1, 1, 1, 1, 0.74, 0.74, 0.73],
    "July 2021":    [1, 1, 1, 1, 0.83, 0.81, 0.81],
    "October 2022": [1, 1, 1, 1, 1.00, 0.85, 0.82]
}, index=index)

Pf = pd.DataFrame({
    "January 2020": [0, 0, 0, 0, 0.26, 0.26, 0.27],
    "July 2021":    [0, 0, 0, 0, 0.17, 0.19, 0.19],
    "October 2022": [0, 0, 0, 0, 0.00, 0.15, 0.18]
}, index=index)

Kst = pd.DataFrame({
    "January 2020": [1.4,2.84,5.32,6.9,3.7,1.63,2.53],
    "July 2021":    [5.83,2.45,7.95,4.91,16.5,3.98,3.62],
    "October 2022": [0.71,0.81,0.72,1.26,2.29,5.42,1.49]
}, index=index)

Ksf = pd.DataFrame({
    "January 2020": [0, 0, 0, 0, 423.13,8.28,43.99],
    "July 2021":    [0, 0, 0, 0, 549.19,94.49,27.54],
    "October 2022": [0, 0, 0, 0, 0.00, 134.4, 6.92]
}, index=index)

# Compute weighted diffusivity contributions for each layer and campaign
K_total = Pt * Kst + Pf * Ksf

# --- Define intervals for upper and lower oxyclines (same as flux data) ---
upper_intervals = {
    "January 2020": ["(2)", "(3)", "(4)"],
    "July 2021":    ["(2)", "(3)", "(4)"],
    "October 2022": ["(2)", "(3)", "(4)", "(5)"]
}

lower_intervals = {
    "January 2020": ["(5)", "(6)", "(7)"],
    "July 2021":    ["(5)", "(6)", "(7)"],
    "October 2022": ["(6)", "(7)"]
}

# Compute the relative contribution of Ksf (salt-finger) to K_total per interval
P_saltfingers = (Pf * Ksf) / K_total * 100  # percentage contribution

# Replace NaN (from divisions by zero) with 0 for clarity
P_saltfingers = P_saltfingers.fillna(0)

# Optional: round for neatness
P_saltfingers = P_saltfingers.round(2)

P_saltfingers



,January 2020,July 2021,October 2022
(1),0.00,0.00,0.00
(2),0.00,0.00,0.00
(3),0.00,0.00,0.00
(4),0.00,0.00,0.00
(5),97.57,87.21,0.00
(6),64.09,84.78,81.40
(7),86.54,64.09,50.48


In [17]:
print("Ksf contribution to K total (%) in the lower oxycline")
print("January 2020 average:")
print((97.57+64.09+86.56)/3)
print("July 2021 average:")
print((87.21+84.78+64.09)/3)
print("October 2022 average:")
print((81.40+50.48)/2)

Ksf contribution to K total (%) in the lower oxycline
January 2020 average:
82.74
July 2021 average:
78.69333333333334
October 2022 average:
65.94
